In [5]:
import xml.etree.ElementTree as ET

for species, accession in [("zea_mays", "E-GEOD-50191"), ("oryza_sativa", "E-MTAB-2037")]:
    path = f"{DATA_DIR}/raw_expression_atlas_{species}/{accession}/{accession}-configuration.xml"
    print(f"\n=== {species} — {accession} ===")
    tree = ET.parse(path)
    root = tree.getroot()
    print("  balise racine :", root.tag, root.attrib)

    # on imprime la structure brute des premiers niveaux pour voir comment c'est organise
    for i, child in enumerate(root.iter()):
        if i > 40:
            break
        print("   ", child.tag, child.attrib, (child.text or "").strip()[:60])


=== zea_mays — E-GEOD-50191 ===
  balise racine : configuration {'experimentType': 'rnaseq_mrna_baseline', 'r_data': '1'}
    configuration {'experimentType': 'rnaseq_mrna_baseline', 'r_data': '1'} 
    analytics {} 
    assay_groups {} 
    assay_group {'id': 'g23', 'label': 'vegetative meristem'} 
    assay {} SRR957481
    assay {} SRR957482
    assay_group {'id': 'g9', 'label': 'internode 6-7'} 
    assay {} SRR957477
    assay {} SRR957475
    assay {} SRR957476
    assay_group {'id': 'g10', 'label': 'internode 7-8'} 
    assay {} SRR957478
    assay {} SRR957479
    assay {} SRR957480
    assay_group {'id': 'g8', 'label': 'growth zone of leaf 8'} 
    assay {} SRR957431
    assay {} SRR957430
    assay {} SRR957432
    assay_group {'id': 'g20', 'label': 'stomatal division zone of leaf 8'} 
    assay {} SRR957427
    assay {} SRR957429
    assay {} SRR957428
    assay_group {'id': 'g21', 'label': 'symmetrical division zone of leaf 8'} 
    assay {} SRR957425
    assay {} SRR95742

In [7]:
import xml.etree.ElementTree as ET
import csv
import statistics

EXPERIMENTS = {
    "zea_mays": "E-GEOD-50191",
    "oryza_sativa": "E-MTAB-2037",
}

dim_protein_path = f"{DATA_DIR}/normalized/dim_protein.csv"

def load_plaza_to_proteins(species):
    """plaza_gene_id -> liste de protein_accession (un gene peut avoir plusieurs isoformes)"""
    mapping = {}
    with open(dim_protein_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["assemblage"] == species and row.get("plaza_gene_id"):
                mapping.setdefault(row["plaza_gene_id"], []).append(row["protein_accession"])
    return mapping

def load_assay_group_labels(species, accession):
    path = f"{DATA_DIR}/raw_expression_atlas_{species}/{accession}/{accession}-configuration.xml"
    tree = ET.parse(path)
    root = tree.getroot()
    return {ag.attrib["id"]: ag.attrib["label"] for ag in root.iter("assay_group")}

def parse_tpm_file(path):
    with open(path, encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        sample_cols = header[2:]  # colonnes 0,1 = GeneID, Gene Name
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 2:
                continue
            gene_id = fields[0]
            values = fields[2:]
            yield gene_id, dict(zip(sample_cols, values))

output_rows = []
for species, accession in EXPERIMENTS.items():
    print(f"traitement {species} / {accession}...")
    plaza_to_proteins = load_plaza_to_proteins(species)
    labels = load_assay_group_labels(species, accession)
    tpm_path = f"{DATA_DIR}/raw_expression_atlas_{species}/{accession}/{accession}-tpms.tsv"

    n_seen, n_bridged = 0, 0
    for gene_id, sample_values in parse_tpm_file(tpm_path):
        n_seen += 1
        proteins = plaza_to_proteins.get(gene_id)
        if not proteins:
            continue
        n_bridged += 1
        for group_id, raw in sample_values.items():
            raw = raw.strip()
            if not raw:
                continue
            reps = [float(x) for x in raw.split(",") if x != ""]
            if not reps:
                continue
            tpm_median = round(statistics.median(reps), 3)
            tissue = labels.get(group_id, group_id)
            for protein_accession in proteins:
                output_rows.append({
                    "assemblage": species,
                    "protein_accession": protein_accession,
                    "plaza_gene_id": gene_id,
                    "tissue": tissue,
                    "tpm_median": tpm_median,
                    "n_replicates": len(reps),
                    "source_experiment": accession,
                })
    print(f"  genes dans le TPM : {n_seen}, pontes vers protein_accession : {n_bridged}")

out_path = f"{DATA_DIR}/normalized/fact_expression.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["assemblage","protein_accession","plaza_gene_id","tissue","tpm_median","n_replicates","source_experiment"])
    writer.writeheader()
    writer.writerows(output_rows)

print(f"\n{len(output_rows)} lignes ecrites dans {out_path}")

traitement zea_mays / E-GEOD-50191...
  genes dans le TPM : 44303, pontes vers protein_accession : 17480
traitement oryza_sativa / E-MTAB-2037...
  genes dans le TPM : 38866, pontes vers protein_accession : 26651

833554 lignes ecrites dans /content/drive/MyDrive/files/data/normalized/fact_expression.csv


In [1]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive"
DRIVE_DIR = ROOT + "/files"
DATA_DIR = DRIVE_DIR + "/data"

for name in ("sources.yaml", "fetch_raw.py"):
    assert os.path.exists(f"{DRIVE_DIR}/{name}"), f"{name} absent de {DRIVE_DIR}"
    shutil.copy(f"{DRIVE_DIR}/{name}", ".")
print("Scripts copies depuis", DRIVE_DIR, "| donnees dans", DATA_DIR)

Mounted at /content/drive
Scripts copies depuis /content/drive/MyDrive/files | donnees dans /content/drive/MyDrive/files/data


In [ ]:
#On commence par le maïs (le plus simple, pont direct via refseq
import csv
from pathlib import Path

PLAZA_DIR = "/content/drive/MyDrive/plaza"

# 1. construire le pont plaza_gene_id <-> protein_accession (NCBI) pour le maïs
zma_bridge = {}  # plaza_gene_id -> set de protein_accession NCBI
with open(f"{PLAZA_DIR}/id_conversion.zma.csv", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 3:
            continue
        gene_id, id_type, ext_id = cols[0], cols[1], cols[2]
        if id_type == "refseq" and (ext_id.startswith("XP_") or ext_id.startswith("NP_")):
            zma_bridge.setdefault(gene_id, set()).add(ext_id)

print(f"Pont maïs construit : {len(zma_bridge)} gènes PLAZA avec au moins un protein_accession NCBI")

# aperçu
for i, (k, v) in enumerate(zma_bridge.items()):
    print(k, v)
    if i >= 5:
        break

Pont maïs construit : 17480 gènes PLAZA avec au moins un protein_accession NCBI
Zm00001eb000170 {'NP_001143585.1'}
Zm00001eb000200 {'NP_001146998.1'}
Zm00001eb000220 {'NP_001136827.1'}
Zm00001eb000230 {'XP_008651020.1', 'NP_001232846.1'}
Zm00001eb000240 {'XP_008677192.1', 'NP_001149551.1'}
Zm00001eb000320 {'XP_008649734.1'}


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

PLAZA_DIR = "/content/drive/MyDrive/plaza"
PFAM_CSV = f"{DATA_DIR}/pfam_scan/zea_mays/pfam_domains_zea_mays.csv"
OUT_CSV = f"{DATA_DIR}/pfam_scan/zea_mays/enriched_zea_mays.csv"

# 1. pont plaza_gene_id <-> protein_accession NCBI
plaza_to_ncbi = defaultdict(set)
ncbi_to_plaza = defaultdict(set)
with open(f"{PLAZA_DIR}/id_conversion.zma.csv", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 3:
            continue
        gene_id, id_type, ext_id = cols[0], cols[1], cols[2]
        if id_type == "refseq" and (ext_id.startswith("XP_") or ext_id.startswith("NP_")):
            plaza_to_ncbi[gene_id].add(ext_id)
            ncbi_to_plaza[ext_id].add(gene_id)

# 2. mapman (classification fonctionnelle)
mapman = defaultdict(list)
with open(f"{PLAZA_DIR}/mapman.zma.csv", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 5:
            continue
        _, _, gene_id, bin_code, desc = cols[:5]
        mapman[gene_id].append(f"{bin_code}:{desc}")

# 3. descriptions
descriptions = defaultdict(list)
with open(f"{PLAZA_DIR}/gene_description.zma.csv", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 3:
            continue
        gene_id, _, desc = cols[:3]
        descriptions[gene_id].append(desc)

# 4. familles orthologues (fichier global, filtré sur zma)
orthofam = {}
with open(f"{PLAZA_DIR}/genefamily_data.ORTHOFAM.csv", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 3:
            continue
        gf_id, species, gene_id = cols[:3]
        if species == "zma":
            orthofam[gene_id] = gf_id

print(f"mapman: {len(mapman)} gènes, descriptions: {len(descriptions)} gènes, orthofam: {len(orthofam)} gènes")

# 5. domaines Pfam par protein_accession
pfam_by_protein = defaultdict(list)
with open(PFAM_CSV, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        pfam_by_protein[row["protein_accession"]].append(f'{row["pfam_name"]}:{row["pfam_accession"]}')

# 6. fusion finale, une ligne par protein_accession
rows_out = []
for protein_accession, plaza_ids in ncbi_to_plaza.items():
    pfam_list = pfam_by_protein.get(protein_accession, [])
    for plaza_id in plaza_ids:
        rows_out.append({
            "protein_accession": protein_accession,
            "plaza_gene_id": plaza_id,
            "pfam_domains": ";".join(pfam_list),
            "gene_description": " | ".join(descriptions.get(plaza_id, [])),
            "mapman": ";".join(mapman.get(plaza_id, [])),
            "orthologous_family": orthofam.get(plaza_id, ""),
        })

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["protein_accession", "plaza_gene_id", "pfam_domains",
                                       "gene_description", "mapman", "orthologous_family"])
    w.writeheader()
    w.writerows(rows_out)

print(f"{len(rows_out)} lignes écrites -> {OUT_CSV}")
print("Aperçu :")
for r in rows_out[:3]:
    print(r)

mapman: 39756 gènes, descriptions: 39756 gènes, orthofam: 39756 gènes
23594 lignes écrites -> /content/drive/MyDrive/files/data/pfam_scan/zea_mays/enriched_zea_mays.csv
Aperçu :
{'protein_accession': 'NP_001143585.1', 'plaza_gene_id': 'Zm00001eb000170', 'pfam_domains': 'MSC:PF09402.17', 'gene_description': 'Zm00001e000017', 'mapman': '35.1:not assigned.annotated', 'orthologous_family': 'ORTHO05D008209'}
{'protein_accession': 'NP_001146998.1', 'plaza_gene_id': 'Zm00001eb000200', 'pfam_domains': '', 'gene_description': 'Zm00001e100035', 'mapman': '5.9.2:Lipid metabolism.lipid droplet-associated activities.lipid droplet structural protein (OLEOSIN/OLO)', 'orthologous_family': 'ORTHO05D000624'}
{'protein_accession': 'NP_001136827.1', 'plaza_gene_id': 'Zm00001eb000220', 'pfam_domains': 'Peptidase_C97:PF05903.21', 'gene_description': 'Zm00001e000022', 'mapman': '19.4.1.7:Protein homeostasis.proteolysis.cysteine-type peptidase activities.C97-class peptidase (PPPDE)', 'orthologous_family': 'OR

In [ ]:
#Voici le script de fusion final, avec les bons chemins (Identifiers/, Descriptions/) et le renommage honnête de la colonne
import csv
from collections import defaultdict
from pathlib import Path

PLAZA_DIR = "/content/drive/MyDrive/plaza monocots"
PFAM_CSV = f"{DATA_DIR}/pfam_scan/zea_mays/pfam_domains_zea_mays.csv"
OUT_CSV = f"{DATA_DIR}/pfam_scan/zea_mays/enriched_zea_mays.csv"

def read_tsv(path, skip_hash=True):
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            if skip_hash and line.startswith("#"):
                continue
            yield line.rstrip("\n").split("\t")

# 1. pont plaza_gene_id <-> protein_accession NCBI (instance Monocots, dossier Identifiers)
plaza_to_ncbi = defaultdict(set)
ncbi_to_plaza = defaultdict(set)
for cols in read_tsv(f"{PLAZA_DIR}/Identifiers/id_conversion.zma.csv"):
    if len(cols) < 3:
        continue
    gene_id, id_type, ext_id = cols[0], cols[1], cols[2]
    if id_type == "refseq" and (ext_id.startswith("XP_") or ext_id.startswith("NP_")):
        plaza_to_ncbi[gene_id].add(ext_id)
        ncbi_to_plaza[ext_id].add(gene_id)

# 2. mapman
mapman = defaultdict(list)
for cols in read_tsv(f"{PLAZA_DIR}/MapMan/mapman.zma.csv"):
    if len(cols) < 5:
        continue
    _, _, gene_id, bin_code, desc = cols[:5]
    mapman[gene_id].append(f"{bin_code}:{desc}")

# 3. "description" -- pour le mais, c'est en realite un ID MaizeGDB historique, pas du texte
# (verifie : 39756/39756 lignes = un ID Zm00001e..., 0 ligne de texte libre)
maizegdb_locus_id = {}
for cols in read_tsv(f"{PLAZA_DIR}/Descriptions/gene_description.zma.csv"):
    if len(cols) < 3:
        continue
    gene_id, _, value = cols[:3]
    maizegdb_locus_id[gene_id] = value

# 4. familles orthologues -- instance Monocots
orthofam = {}
for cols in read_tsv(f"{PLAZA_DIR}/Gene Families Data/genefamily_data.ORTHOFAM.csv"):
    if len(cols) < 3:
        continue
    gf_id, species, gene_id = cols[:3]
    if species == "zma":
        orthofam[gene_id] = gf_id

# 5. InterPro
interpro = defaultdict(list)
for cols in read_tsv(f"{PLAZA_DIR}/InterPro/interpro.zma.csv"):
    if len(cols) < 4:
        continue
    gene_id, _, motif_id, desc = cols[0], cols[1], cols[2], cols[3]
    interpro[gene_id].append(f"{motif_id}:{desc}")

# 6. GO (PLAZA)
go_terms = defaultdict(list)
for cols in read_tsv(f"{PLAZA_DIR}/Gene Ontology/go.zma.csv"):
    if len(cols) < 9:
        continue
    gene_id, _, go, evidence = cols[0], cols[1], cols[2], cols[3]
    go_terms[gene_id].append(f"{go}({evidence})")

# 7. coordonnees -- all_transcripts, reduites a 1 ligne/gene (min start, max stop)
coords = {}
for cols in read_tsv(f"{PLAZA_DIR}/Structural Annotation/annotation.all_transcripts.zma.csv"):
    if len(cols) < 10:
        continue
    gene_id, start, stop, strand, chr_ = cols[0], cols[4], cols[5], cols[8], cols[9]
    try:
        start, stop = int(start), int(stop)
    except ValueError:
        continue
    if gene_id not in coords:
        coords[gene_id] = [chr_, start, stop, strand]
    else:
        coords[gene_id][1] = min(coords[gene_id][1], start)
        coords[gene_id][2] = max(coords[gene_id][2], stop)

# 8. domaines Pfam par protein_accession (Phase 3)
pfam_by_protein = defaultdict(list)
with open(PFAM_CSV, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        pfam_by_protein[row["protein_accession"]].append(f'{row["pfam_name"]}:{row["pfam_accession"]}')

# 9. fusion finale
rows_out = []
for protein_accession, plaza_ids in ncbi_to_plaza.items():
    pfam_list = pfam_by_protein.get(protein_accession, [])
    for plaza_id in plaza_ids:
        c = coords.get(plaza_id, ["", "", "", ""])
        rows_out.append({
            "protein_accession": protein_accession,
            "plaza_gene_id": plaza_id,
            "chr": c[0], "start": c[1], "stop": c[2], "strand": c[3],
            "pfam_domains": ";".join(pfam_list),
            "interpro_domains": ";".join(interpro.get(plaza_id, [])),
            "go_terms": ";".join(go_terms.get(plaza_id, [])),
            "maizegdb_locus_id": maizegdb_locus_id.get(plaza_id, ""),
            "mapman": ";".join(mapman.get(plaza_id, [])),
            "orthologous_family": orthofam.get(plaza_id, ""),
        })

fieldnames = ["protein_accession", "plaza_gene_id", "chr", "start", "stop", "strand",
              "pfam_domains", "interpro_domains", "go_terms", "maizegdb_locus_id",
              "mapman", "orthologous_family"]
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows_out)

print(f"{len(rows_out)} lignes ecrites")
print(f"familles orthologues Monocots : {sum(1 for r in rows_out if r['orthologous_family'].startswith('ORTHO05M'))}/{len(rows_out)}")
print(f"pont id_conversion zma (instance verifiee) : {len(ncbi_to_plaza)} protein_accession NCBI pontes")

23594 lignes ecrites
familles orthologues Monocots : 23594/23594
pont id_conversion zma (instance verifiee) : 22975 protein_accession NCBI pontes


In [ ]:
from collections import Counter

PLAZA_DIR = "/content/drive/MyDrive/plaza monocots"
path = f"{PLAZA_DIR}/Identifiers/id_conversion.osa.csv"

id_types = Counter()
uniprot_samples = []

with open(path, encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 3:
            continue
        gene_id, id_type, ext_id = cols[:3]
        id_types[id_type] += 1
        if id_type == "uniprot" and len(uniprot_samples) < 10:
            uniprot_samples.append((gene_id, ext_id))

print("types d'ID presents et leur frequence :")
for t, c in id_types.most_common():
    print(f"  {t}: {c}")

print("\nexemples uniprot :")
for g, u in uniprot_samples:
    print(f"  {g} -> {u}")

types d'ID presents et leur frequence :
  MSU-ID: 40076
  tid: 37870
  uniprot: 35656
  symbol-oryzabase: 21771
  symbol: 6468
  name-CGSNL: 6421
  symbol-CGSNL: 5506

exemples uniprot :
  Os01g0100100 -> Q0JRI1
  Os01g0100200 -> B9EYQ4
  Os01g0100300 -> A0A0P0UWT2
  Os01g0100400 -> Q2HPI7
  Os01g0100466 -> A0A0P0UXH5
  Os01g0100500 -> A2ZN82
  Os01g0100600 -> Q2HPI5
  Os01g0100650 -> A0A0P0UX34
  Os01g0100700 -> Q93VC6
  Os01g0100800 -> B9EYQ7


In [ ]:
#(Phase 1) pour obtenir le protein_accession.
import requests, time
from collections import defaultdict

PLAZA_DIR = "/content/drive/MyDrive/plaza monocots"

# 1. gene_id PLAZA -> accession(s) UniProt
plaza_to_uniprot = defaultdict(set)
with open(f"{PLAZA_DIR}/Identifiers/id_conversion.osa.csv", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 3:
            continue
        gene_id, id_type, ext_id = cols[:3]
        if id_type == "uniprot":
            plaza_to_uniprot[gene_id].add(ext_id)

all_accessions = sorted({acc for accs in plaza_to_uniprot.values() for acc in accs})
print(f"genes PLAZA avec >=1 uniprot : {len(plaza_to_uniprot)}")
print(f"accessions uniprot distinctes : {len(all_accessions)}")

# 2. soumission du job UniProt ID Mapping : UniProtKB accession -> GeneID (Entrez)
submit = requests.post(
    "https://rest.uniprot.org/idmapping/run",
    data={"from": "UniProtKB_AC-ID", "to": "GeneID", "ids": ",".join(all_accessions)},
)
submit.raise_for_status()
job_id = submit.json()["jobId"]
print("job UniProt soumis :", job_id)

# 3. attente de fin de job
status_url = f"https://rest.uniprot.org/idmapping/status/{job_id}"
while True:
    r = requests.get(status_url)
    data = r.json()
    if "results" in data or "failedIds" in data or data.get("jobStatus") in (None, "FINISHED"):
        break
    print("  en cours...", data.get("jobStatus"))
    time.sleep(5)

# 4. recuperation des resultats (pagine automatiquement par le client si besoin, ici direct)
uniprot_to_geneid = {}
results_url = f"https://rest.uniprot.org/idmapping/uniprotkb/results/stream/{job_id}?fields=accession&format=tsv"
# le mapping brut (accession -> GeneID) est plus simple via l'endpoint "results" non-uniprotkb :
results_url = f"https://rest.uniprot.org/idmapping/stream/{job_id}"
r = requests.get(results_url)
r.raise_for_status()
for entry in r.json().get("results", []):
    uniprot_to_geneid[entry["from"]] = entry["to"]

print(f"accessions uniprot pontees vers un GeneID : {len(uniprot_to_geneid)} / {len(all_accessions)}")

genes PLAZA avec >=1 uniprot : 35656
accessions uniprot distinctes : 35656
job UniProt soumis : lwGUlfNYuz
  en cours... RUNNING
  en cours... RUNNING
  en cours... RUNNING
  en cours... RUNNING
accessions uniprot pontees vers un GeneID : 933 / 35656


In [ ]:
import gzip, csv
from collections import defaultdict

PLAZA_DIR = "/content/drive/MyDrive/plaza monocots"
# A ajuster selon ton chemin reel de la Phase 1 pour le riz
NCBI_FEATURE_TABLE = f"{DATA_DIR}/raw/oryza_sativa/ncbi_refseq_oryza_irgsp/IRGSP-1.0_GCF_001433935.1_verifier_l_assemblage_RefSeq_coura/GCF_001433935.1_IRGSP-1.0_feature_table.txt.gz"

# 1. coordonnees PLAZA riz, reduites a 1 ligne/gene
plaza_coords = {}
with open(f"{PLAZA_DIR}/Structural Annotation/annotation.all_transcripts.osa.csv", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 10:
            continue
        gene_id, start, stop, strand, chr_ = cols[0], cols[4], cols[5], cols[8], cols[9]
        try:
            start, stop = int(start), int(stop)
        except ValueError:
            continue
        if gene_id not in plaza_coords:
            plaza_coords[gene_id] = [chr_, start, stop, strand]
        else:
            plaza_coords[gene_id][1] = min(plaza_coords[gene_id][1], start)
            plaza_coords[gene_id][2] = max(plaza_coords[gene_id][2], stop)

print(f"genes PLAZA riz avec coordonnees : {len(plaza_coords)}")

# 2. genes NCBI (Phase 1), indexes par chromosome, tries par position
ncbi_genes = defaultdict(list)
with gzip.open(NCBI_FEATURE_TABLE, "rt", encoding="utf-8", errors="replace") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        if row.get("# feature") != "gene":
            continue
        chrom = row.get("chromosome", "")
        try:
            start, end = int(row["start"]), int(row["end"])
        except (ValueError, KeyError):
            continue
        strand = row.get("strand", "")
        geneid = row.get("GeneID", "")
        if not chrom or not geneid:
            continue
        ncbi_genes[chrom].append((start, end, strand, geneid))

for chrom in ncbi_genes:
    ncbi_genes[chrom].sort()

print("chromosomes NCBI trouves :", sorted(ncbi_genes.keys())[:15])
print("chromosomes PLAZA trouves :", sorted({v[0] for v in plaza_coords.values()})[:15])

# 3. pont par chevauchement de position (meme chromosome, meme brin, intervalle qui se recoupe)
def normalize_chr(chrom):
    # "chr01" -> "1", "chr12" -> "12"
    if chrom.startswith("chr"):
        return str(int(chrom[3:]))
    return chrom

def find_overlap(chrom, start, stop, strand):
    chrom = normalize_chr(chrom)
    best, best_ov = None, 0
    for nstart, nend, nstrand, geneid in ncbi_genes.get(chrom, []):
        if nstart > stop:
            break
        if nstrand != strand:
            continue
        ov = min(stop, nend) - max(start, nstart)
        if ov > best_ov:
            best_ov, best = ov, geneid
    return best

plaza_to_ncbi_geneid = {}
for gene_id, (chr_, start, stop, strand) in plaza_coords.items():
    gid = find_overlap(chr_, start, stop, strand)
    if gid:
        plaza_to_ncbi_geneid[gene_id] = gid

print(f"genes PLAZA pontes par position vers un GeneID NCBI : {len(plaza_to_ncbi_geneid)} / {len(plaza_coords)}")

plaza_to_proteins = {g: geneid_to_proteins.get(gid, set()) for g, gid in plaza_to_ncbi_geneid.items()}
n_with_protein = sum(1 for p in plaza_to_proteins.values() if p)
print(f"genes PLAZA avec au moins 1 protein_accession au bout du pont : {n_with_protein}")

genes PLAZA riz avec coordonnees : 37870
chromosomes NCBI trouves : ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9', 'MT', 'Pltd']
chromosomes PLAZA trouves : ['chr01', 'chr02', 'chr03', 'chr04', 'chr05', 'chr06', 'chr07', 'chr08', 'chr09', 'chr10', 'chr11', 'chr12']
genes PLAZA pontes par position vers un GeneID NCBI : 29328 / 37870
genes PLAZA avec au moins 1 protein_accession au bout du pont : 26685


In [ ]:
# construire enriched_oryza_sativa.csv, avec les mêmes couches que le maïs. Script de fusion complet :
import csv
from collections import defaultdict

PLAZA_DIR = "/content/drive/MyDrive/plaza monocots"
PFAM_CSV = f"{DATA_DIR}/pfam_scan/oryza_sativa/pfam_domains_oryza_sativa.csv"
OUT_CSV = f"{DATA_DIR}/pfam_scan/oryza_sativa/enriched_oryza_sativa.csv"

def read_tsv(path):
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("#"):
                continue
            yield line.rstrip("\n").split("\t")

# mapman
mapman = defaultdict(list)
for cols in read_tsv(f"{PLAZA_DIR}/MapMan/mapman.osa.csv"):
    if len(cols) < 5:
        continue
    _, _, gene_id, bin_code, desc = cols[:5]
    mapman[gene_id].append(f"{bin_code}:{desc}")

# description (vrai texte pour le riz, verifie precedemment)
descriptions = defaultdict(list)
for cols in read_tsv(f"{PLAZA_DIR}/Descriptions/gene_description.osa.csv"):
    if len(cols) < 3:
        continue
    gene_id, _, desc = cols[:3]
    descriptions[gene_id].append(desc)

# orthologie (instance Monocots)
orthofam = {}
for cols in read_tsv(f"{PLAZA_DIR}/Gene Families Data/genefamily_data.ORTHOFAM.csv"):
    if len(cols) < 3:
        continue
    gf_id, species, gene_id = cols[:3]
    if species == "osa":
        orthofam[gene_id] = gf_id

# interpro
interpro = defaultdict(list)
for cols in read_tsv(f"{PLAZA_DIR}/InterPro/interpro.osa.csv"):
    if len(cols) < 4:
        continue
    gene_id, _, motif_id, desc = cols[0], cols[1], cols[2], cols[3]
    interpro[gene_id].append(f"{motif_id}:{desc}")

# GO
go_terms = defaultdict(list)
for cols in read_tsv(f"{PLAZA_DIR}/Gene Ontology/go.osa.csv"):
    if len(cols) < 9:
        continue
    gene_id, _, go, evidence = cols[0], cols[1], cols[2], cols[3]
    go_terms[gene_id].append(f"{go}({evidence})")

# pfam (Phase 3)
pfam_by_protein = defaultdict(list)
with open(PFAM_CSV, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        pfam_by_protein[row["protein_accession"]].append(f'{row["pfam_name"]}:{row["pfam_accession"]}')

# fusion -- via le pont par position construit juste avant (plaza_to_proteins, plaza_coords)
rows_out = []
for plaza_id, proteins in plaza_to_proteins.items():
    if not proteins:
        continue
    chr_, start, stop, strand = plaza_coords[plaza_id]
    for protein_accession in proteins:
        rows_out.append({
            "protein_accession": protein_accession,
            "plaza_gene_id": plaza_id,
            "chr": chr_, "start": start, "stop": stop, "strand": strand,
            "pfam_domains": ";".join(pfam_by_protein.get(protein_accession, [])),
            "interpro_domains": ";".join(interpro.get(plaza_id, [])),
            "go_terms": ";".join(go_terms.get(plaza_id, [])),
            "gene_description": " | ".join(descriptions.get(plaza_id, [])),
            "mapman": ";".join(mapman.get(plaza_id, [])),
            "orthologous_family": orthofam.get(plaza_id, ""),
        })

fieldnames = ["protein_accession", "plaza_gene_id", "chr", "start", "stop", "strand",
              "pfam_domains", "interpro_domains", "go_terms", "gene_description",
              "mapman", "orthologous_family"]
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows_out)

print(f"{len(rows_out)} lignes ecrites")
print(f"familles orthologues Monocots : {sum(1 for r in rows_out if r['orthologous_family'].startswith('ORTHO05M'))}/{len(rows_out)}")

41610 lignes ecrites
familles orthologues Monocots : 40789/41610


In [ ]:
import csv, random

with open(OUT_CSV, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

sample = random.sample(rows, 3)
for r in sample:
    print(r)
    print()

for col in ["chr", "interpro_domains", "go_terms", "gene_description", "orthologous_family"]:
    filled = sum(1 for r in rows if r[col])
    print(f"{col}: {filled}/{len(rows)} ({filled/len(rows)*100:.1f}%)")

{'protein_accession': 'XP_015635388.1', 'plaza_gene_id': 'Os01g0582400', 'chr': 'chr01', 'start': '22591771', 'stop': '22596057', 'strand': '-', 'pfam_domains': 'Pro_isomerase:PF00160.27', 'interpro_domains': 'IPR029000:Cyclophilin-like domain superfamily;IPR044666:Cyclophilin-type peptidyl-prolyl cis-trans isomerase, cyclophilin A-like;IPR002130:Cyclophilin-type peptidyl-prolyl cis-trans isomerase domain;IPR029000:Cyclophilin-like domain superfamily;IPR002130:Cyclophilin-type peptidyl-prolyl cis-trans isomerase domain;IPR002130:Cyclophilin-type peptidyl-prolyl cis-trans isomerase domain;IPR002130:Cyclophilin-type peptidyl-prolyl cis-trans isomerase domain;IPR002130:Cyclophilin-type peptidyl-prolyl cis-trans isomerase domain', 'go_terms': 'GO:0009506(ISO);GO:0005575(ISO);GO:0005911(ISO);GO:0030054(ISO);GO:0055044(ISO);GO:0070161(ISO);GO:0110165(ISO);GO:0000413(IEA);GO:0006464(IEA);GO:0006807(IEA);GO:0008150(IEA);GO:0008152(IEA);GO:0009987(IEA);GO:0018193(IEA);GO:0018208(IEA);GO:0019538

In [ ]:
import gzip

path = "/content/drive/MyDrive/plaza monocots/reference/uniprot_sprot.dat.gz"  # ajuste si le chemin exact differe

with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 5:
            break

ID   001R_FRG3G              Reviewed;         256 AA.
AC   Q6GZX4;
DT   28-JUN-2011, integrated into UniProtKB/Swiss-Prot.
DT   19-JUL-2004, sequence version 1.
DT   02-SEP-2026, entry version 48.
DE   RecName: Full=Putative transcription factor 001R;


In [ ]:
import os

# Recherche des fichiers fasta (.faa, .fa, .fasta) associés à 'tn90' et 'k326' dans DATA_DIR
target_keywords = ['tn90', 'k326']
found_paths = []

print(f"Recherche en cours dans {DATA_DIR}...")
for root, dirs, files in os.walk(DATA_DIR):
    # Ignorer les dossiers cachés
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for file in files:
        file_lower = file.lower()
        if any(ext in file_lower for ext in ['.faa', '.fa', '.fasta']):
            if any(keyword in file_lower or keyword in root.lower() for keyword in target_keywords):
                full_path = os.path.join(root, file)
                found_paths.append(full_path)
                print(f"Trouvé : {full_path}")

if not found_paths:
    print("Aucun fichier d'extension protéine trouvé directement sous DATA_DIR avec ces mots-clés. Élargissement de la recherche à DRIVE_DIR...")
    for root, dirs, files in os.walk(DRIVE_DIR):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        for file in files:
            file_lower = file.lower()
            if any(ext in file_lower for ext in ['.faa', '.fa', '.fasta']):
                if any(keyword in file_lower or keyword in root.lower() for keyword in target_keywords):
                    full_path = os.path.join(root, file)
                    print(f"Trouvé : {full_path}")

Recherche en cours dans /content/drive/MyDrive/files/data...
Trouvé : /content/drive/MyDrive/files/data/raw/nicotiana_tabacum/ncbi_refseq_nitab_tn90/Ntab-TN90_GCF_000715135.1/GCF_000715135.1_Ntab-TN90_protein.faa.gz
Trouvé : /content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_tn90/chunks/chunk_00000.faa
Trouvé : /content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_tn90/chunks/chunk_00001.faa
Trouvé : /content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_tn90/chunks/chunk_00002.faa
Trouvé : /content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_tn90/chunks/chunk_00003.faa
Trouvé : /content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_tn90/chunks/chunk_00004.faa
Trouvé : /content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_tn90/chunks/chunk_00005.faa
Trouvé : /content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_tn90/chunks/chunk_00006.faa
Trouvé : /content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_tn90/chunks/chunk_00

In [ ]:
!wget -q https://github.com/bbuchfink/diamond/releases/latest/download/diamond-linux64.tar.gz
!tar xzf diamond-linux64.tar.gz
!./diamond --version

diamond version 2.2.8


In [ ]:
#Script d’alignement, TN90 puis K326 (à adapter : remplace les deux chemins TODO par les vrais chemins de tes protein.faa Phase 1) :
import subprocess
import os

DIAMOND_BIN = "./diamond"
SWISSPROT_DMND = "/content/drive/MyDrive/plaza monocots/reference/swissprot_diamond.dmnd"

targets = {
    "tn90": "/content/drive/MyDrive/files/data/raw/nicotiana_tabacum/ncbi_refseq_nitab_tn90/Ntab-TN90_GCF_000715135.1/GCF_000715135.1_Ntab-TN90_protein.faa.gz",
    "k326": "/content/drive/MyDrive/files/data/raw/nicotiana_tabacum/ncbi_refseq_nitab_asm71507v2/ASM71507v2_GCF_000715075.1_RS_2025_07/GCF_000715075.1_ASM71507v2_protein.faa.gz"
}

os.makedirs(f"{DATA_DIR}/diamond", exist_ok=True)

for name, faa_path in targets.items():
    out_tsv = f"{DATA_DIR}/diamond/{name}_vs_swissprot.tsv"
    print(f"--- {name} ---")
    subprocess.run([
        DIAMOND_BIN, "blastp",
        "--query", faa_path,
        "--db", SWISSPROT_DMND,
        "--out", out_tsv,
        "--outfmt", "6", "qseqid", "sseqid", "pident", "length", "evalue", "bitscore",
        "--max-target-seqs", "1",
        "--evalue", "1e-5",
        "--threads", "4",
    ], check=True)
    n_lines = sum(1 for _ in open(out_tsv))
    print(f"{name} : {n_lines} hits -> {out_tsv}")

--- tn90 ---
tn90 : 60627 hits -> /content/drive/MyDrive/files/data/diamond/tn90_vs_swissprot.tsv
--- k326 ---
k326 : 64776 hits -> /content/drive/MyDrive/files/data/diamond/k326_vs_swissprot.tsv


In [ ]:
!wget -q "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.dat.gz" -O /content/uniprot_sprot_new.dat.gz
!gzip -t /content/uniprot_sprot_new.dat.gz && echo "fichier valide"
!ls -lh /content/uniprot_sprot_new.dat.gz

fichier valide
-rw-r--r-- 1 root root 668M Sep  3 18:00 /content/uniprot_sprot_new.dat.gz


In [ ]:
import gzip, csv
from collections import defaultdict

# 1. retelecharger swiss-prot (ephemere sur /content, perdu au redemarrage)
!wget -q "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.dat.gz" -O /content/uniprot_sprot_new.dat.gz
!gzip -t /content/uniprot_sprot_new.dat.gz && echo "fichier valide"

SWISSPROT_DAT = "/content/uniprot_sprot_new.dat.gz"

# 2. recharger les hits DIAMOND (sur Drive, toujours la)
def load_best_hits(tsv_path):
    best = {}
    with open(tsv_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            cols = line.rstrip("\n").split("\t")
            qseqid, sseqid = cols[0], cols[1]
            if qseqid not in best:
                best[qseqid] = sseqid
    return best

tn90_hits = load_best_hits(f"{DATA_DIR}/diamond/tn90_vs_swissprot.tsv")
k326_hits = load_best_hits(f"{DATA_DIR}/diamond/k326_vs_swissprot.tsv")
needed_accessions = set(tn90_hits.values()) | set(k326_hits.values())
print(f"accessions necessaires : {len(needed_accessions)}")

# 3. reparser swiss-prot
def extract_accession(sseqid):
    parts = sseqid.split("|")
    return parts[1] if len(parts) >= 2 else sseqid

needed_accessions = {extract_accession(v) for v in list(tn90_hits.values()) + list(k326_hits.values())}
print(f"accessions necessaires (format corrige) : {len(needed_accessions)}")

swissprot_info = {}
with gzip.open(SWISSPROT_DAT, "rt", encoding="utf-8", errors="replace") as f:
    current_ac, current_de, current_go = None, [], []
    for line in f:
        if line.startswith("AC   "):
            if current_ac is None:
                current_ac = line[5:].split(";")[0].strip()
        elif line.startswith("DE   "):
            if "Full=" in line:
                current_de.append(line.split("Full=")[1].split(";")[0].split("{")[0].strip())
        elif line.startswith("DR   GO;"):
            parts = line.split(";")
            go_id = parts[1].strip()
            go_cat_desc = parts[2].strip() if len(parts) > 2 else ""
            current_go.append(f"{go_id}:{go_cat_desc}")
        elif line.startswith("//"):
            if current_ac in needed_accessions:
                swissprot_info[current_ac] = {
                    "description": " | ".join(current_de),
                    "go_terms": ";".join(current_go),
                }
            current_ac, current_de, current_go = None, [], []

print(f"accessions trouvees : {len(swissprot_info)} / {len(needed_accessions)}")

def build_enriched(hits, pfam_csv, out_csv):
    pfam_by_protein = defaultdict(list)
    with open(pfam_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pfam_by_protein[row["protein_accession"]].append(f'{row["pfam_name"]}:{row["pfam_accession"]}')
    rows_out = []
    for protein_accession, sp_accession in hits.items():
        info = swissprot_info.get(extract_accession(sp_accession), {"description": "", "go_terms": ""})
        rows_out.append({
            "protein_accession": protein_accession,
            "swissprot_accession": extract_accession(sp_accession),
            "pfam_domains": ";".join(pfam_by_protein.get(protein_accession, [])),
            "swissprot_description": info["description"],
            "swissprot_go_terms": info["go_terms"],
        })
    fieldnames = ["protein_accession", "swissprot_accession", "pfam_domains",
                  "swissprot_description", "swissprot_go_terms"]
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows_out)
    print(f"{out_csv} : {len(rows_out)} lignes")

build_enriched(tn90_hits,
    "/content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_tn90/pfam_domains_nicotiana_tabacum.csv",
    f"{DATA_DIR}/diamond/enriched_nicotiana_tabacum_tn90.csv")

build_enriched(k326_hits,
    "/content/drive/MyDrive/files/data/pfam_scan/nicotiana_tabacum_k326/pfam_domains_nicotiana_tabacum.csv",
    f"{DATA_DIR}/diamond/enriched_nicotiana_tabacum_k326.csv")

fichier valide
accessions necessaires : 13990
accessions necessaires (format corrige) : 13990
accessions trouvees : 13990 / 13990
/content/drive/MyDrive/files/data/diamond/enriched_nicotiana_tabacum_tn90.csv : 60627 lignes
/content/drive/MyDrive/files/data/diamond/enriched_nicotiana_tabacum_k326.csv : 64776 lignes


In [ ]:
#13 990/13 990 trouvées — le tabac est maintenant complet sur les 2 cultivars. Même contrôle qualité qu’avant les deux autres espèces, pour clore proprement :
import csv, random

for name, path in [("TN90", f"{DATA_DIR}/diamond/enriched_nicotiana_tabacum_tn90.csv"),
                    ("K326", f"{DATA_DIR}/diamond/enriched_nicotiana_tabacum_k326.csv")]:
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    print(f"=== {name} ({len(rows)} lignes) ===")
    for r in random.sample(rows, 2):
        print(r)
    for col in ["pfam_domains", "swissprot_description", "swissprot_go_terms"]:
        filled = sum(1 for r in rows if r[col])
        print(f"{col}: {filled}/{len(rows)} ({filled/len(rows)*100:.1f}%)")
    print()

=== TN90 (60627 lignes) ===
{'protein_accession': 'XP_016501889.1', 'swissprot_accession': 'Q9M0Y6', 'pfam_domains': 'Inositol_P:PF00459.31', 'swissprot_description': "Putative 3'(2'),5'-bisphosphate nucleotidase, mitochondrial | 3'(2'),5-bisphosphonucleoside 3'(2')-phosphohydrolase | DPNPase", 'swissprot_go_terms': "GO:0009507:C:chloroplast;GO:0005739:C:mitochondrion;GO:0005634:C:nucleus;GO:0008441:F:3'(2'),5'-bisphosphate nucleotidase activity;GO:0000103:P:sulfate assimilation"}
{'protein_accession': 'XP_016475352.1', 'swissprot_accession': 'Q5E930', 'pfam_domains': 'SLAC1:PF03595.23', 'swissprot_description': 'S-type anion channel SLAH1 | SLAC1-homolog protein 1', 'swissprot_go_terms': 'GO:0005886:C:plasma membrane;GO:0008308:F:voltage-gated monoatomic anion channel activity;GO:0006821:P:chloride transport;GO:0006873:P:intracellular monoatomic ion homeostasis;GO:0009651:P:response to salt stress;GO:0009414:P:response to water deprivation'}
pfam_domains: 57723/60627 (95.2%)
swissprot

In [ ]:
#Voici le script de normalisation — il transforme les 4 fichiers enriched_*.csv (zea_mays, oryza_sativa, nicotiana_tabacum_tn90, nicotiana_tabacum_k326) en 3 tables relationnelles : dim_protein, fact_functional_annotation, fact_orthology.

#Choix de grain : la clé de base est (assemblage, protein_accession) plutôt que le gène seul — c’est le niveau où Phase 3 (Pfam) opère déjà, et ça évite de perdre l’info isoforme. plaza_gene_id reste disponible comme colonne pour regrouper par gène quand besoin (maïs/riz seulement, pas tabac).
import csv
from collections import defaultdict
from pathlib import Path

OUT_DIR = f"{DATA_DIR}/normalized"
Path(OUT_DIR).mkdir(exist_ok=True)

# config par assemblage : quels fichiers, quelles colonnes mappent vers quelle "source"
SPECIES_CONFIG = {
    "zea_mays": {
        "path": f"{DATA_DIR}/pfam_scan/zea_mays/enriched_zea_mays.csv",
        "plaza_gene_id_col": "plaza_gene_id",
        "coord_cols": ("chr", "start", "stop", "strand"),
        "orthofam_col": "orthologous_family",
        "multi_sources": {
            "pfam_domains": "pfam",
            "interpro_domains": "interpro",
            "go_terms": "go_plaza",
            "mapman": "mapman",
        },
        "single_sources": {
            "maizegdb_locus_id": "maizegdb_locus_id",
        },
    },
    "oryza_sativa": {
        "path": f"{DATA_DIR}/pfam_scan/oryza_sativa/enriched_oryza_sativa.csv",
        "plaza_gene_id_col": "plaza_gene_id",
        "coord_cols": ("chr", "start", "stop", "strand"),
        "orthofam_col": "orthologous_family",
        "multi_sources": {
            "pfam_domains": "pfam",
            "interpro_domains": "interpro",
            "go_terms": "go_plaza",
            "mapman": "mapman",
        },
        "single_sources": {
            "gene_description": "description",
        },
    },
    "nicotiana_tabacum_tn90": {
        "path": f"{DATA_DIR}/diamond/enriched_nicotiana_tabacum_tn90.csv",
        "plaza_gene_id_col": None,
        "coord_cols": None,
        "orthofam_col": None,
        "multi_sources": {
            "pfam_domains": "pfam",
            "swissprot_go_terms": "go_swissprot",
        },
        "single_sources": {
            "swissprot_description": "description",
        },
    },
    "nicotiana_tabacum_k326": {
        "path": f"{DATA_DIR}/diamond/enriched_nicotiana_tabacum_k326.csv",
        "plaza_gene_id_col": None,
        "coord_cols": None,
        "orthofam_col": None,
        "multi_sources": {
            "pfam_domains": "pfam",
            "swissprot_go_terms": "go_swissprot",
        },
        "single_sources": {
            "swissprot_description": "description",
        },
    },
}

def parse_multi(value, source):
    """'name:id;name:id' ou 'GO:xxx(evidence);...' ou 'GO:xxx:C:desc;...' -> liste de (term_id, term_desc, evidence)"""
    out = []
    if not value:
        return out
    for part in value.split(";"):
        part = part.strip()
        if not part:
            continue
        if source == "go_plaza":
            # format PLAZA : GO:0005634(IEA)
            if "(" in part:
                go_id, evidence = part.rstrip(")").split("(")
                out.append((go_id, "", evidence))
            else:
                out.append((part, "", ""))
        elif source == "go_swissprot":
            # format Swiss-Prot : GO:0009507:C:chloroplast
            segs = part.split(":", 2)
            if len(segs) == 3:
                go_id = f"{segs[0]}:{segs[1]}"
                aspect, desc = segs[2].split(":", 1) if ":" in segs[2] else (segs[2], "")
                out.append((go_id, desc, ""))
            else:
                out.append((part, "", ""))
        else:
            # format standard : nom:id ou id:desc
            segs = part.split(":", 1)
            if len(segs) == 2:
                out.append((segs[1], segs[0], ""))
            else:
                out.append((part, "", ""))
    return out

dim_protein_rows = []
fact_annotation_rows = []
fact_orthology_rows = []

for assemblage, cfg in SPECIES_CONFIG.items():
    with open(cfg["path"], newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            protein_accession = row["protein_accession"]

            plaza_gene_id = row.get(cfg["plaza_gene_id_col"], "") if cfg["plaza_gene_id_col"] else ""
            if cfg["coord_cols"]:
                chr_, start, stop, strand = (row.get(c, "") for c in cfg["coord_cols"])
            else:
                chr_, start, stop, strand = "", "", "", ""

            dim_protein_rows.append({
                "assemblage": assemblage,
                "protein_accession": protein_accession,
                "plaza_gene_id": plaza_gene_id,
                "chr": chr_, "start": start, "stop": stop, "strand": strand,
            })

            for col, source in cfg["multi_sources"].items():
                for term_id, term_desc, evidence in parse_multi(row.get(col, ""), source):
                    fact_annotation_rows.append({
                        "assemblage": assemblage,
                        "protein_accession": protein_accession,
                        "source": source,
                        "term_id": term_id,
                        "term_desc": term_desc,
                        "evidence": evidence,
                    })

            for col, source in cfg["single_sources"].items():
                val = row.get(col, "")
                if val:
                    fact_annotation_rows.append({
                        "assemblage": assemblage,
                        "protein_accession": protein_accession,
                        "source": source,
                        "term_id": "",
                        "term_desc": val,
                        "evidence": "",
                    })

            if cfg["orthofam_col"]:
                orthofam = row.get(cfg["orthofam_col"], "")
                if orthofam:
                    fact_orthology_rows.append({
                        "assemblage": assemblage,
                        "protein_accession": protein_accession,
                        "plaza_gene_id": plaza_gene_id,
                        "orthofam_id": orthofam,
                        "plaza_instance": "monocots_05",
                    })

def write_csv(path, rows, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"{path} : {len(rows)} lignes")

write_csv(f"{OUT_DIR}/dim_protein.csv", dim_protein_rows,
          ["assemblage", "protein_accession", "plaza_gene_id", "chr", "start", "stop", "strand"])
write_csv(f"{OUT_DIR}/fact_functional_annotation.csv", fact_annotation_rows,
          ["assemblage", "protein_accession", "source", "term_id", "term_desc", "evidence"])
write_csv(f"{OUT_DIR}/fact_orthology.csv", fact_orthology_rows,
          ["assemblage", "protein_accession", "plaza_gene_id", "orthofam_id", "plaza_instance"])

/content/drive/MyDrive/files/data/normalized/dim_protein.csv : 190607 lignes
/content/drive/MyDrive/files/data/normalized/fact_functional_annotation.csv : 7843548 lignes
/content/drive/MyDrive/files/data/normalized/fact_orthology.csv : 64383 lignes


In [ ]:
#Vérifions que le schéma répond bien aux usages prévus avec deux requêtes test :
import csv
from collections import Counter, defaultdict

with open(f"{OUT_DIR}/fact_functional_annotation.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    source_counts = Counter()
    evidence_counts = Counter()
    for row in reader:
        source_counts[row["source"]] += 1
        if row["source"] == "go_plaza":
            evidence_counts[row["evidence"]] += 1

print("repartition par source :")
for s, c in source_counts.most_common():
    print(f"  {s}: {c}")

print("\nrepartition des evidences GO (PLAZA, maïs+riz) :")
for e, c in evidence_counts.most_common():
    print(f"  {e}: {c}")

# test requete : genes dans une famille orthologue partagee entre mais et riz
with open(f"{OUT_DIR}/fact_orthology.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fam_species = defaultdict(set)
    for row in reader:
        sp = "zea_mays" if row["assemblage"] == "zea_mays" else "oryza_sativa"
        fam_species[row["orthofam_id"]].add(sp)

shared = sum(1 for sp_set in fam_species.values() if len(sp_set) == 2)
print(f"\nfamilles orthologues presentes a la fois chez mais ET riz : {shared} / {len(fam_species)}")

repartition par source :
  go_plaza: 5963014
  go_swissprot: 799692
  pfam: 518561
  interpro: 304808
  description: 166991
  mapman: 66888
  maizegdb_locus_id: 23594

repartition des evidences GO (PLAZA, maïs+riz) :
  IEA: 3139546
  ISO: 1793012
  IBA: 952235
  IDA: 28637
  IMP: 24012
  ISS: 8478
  IC: 7767
  IEP: 3906
  TAS: 2911
  IPI: 1724
  IGI: 357
  RCA: 286
  NAS: 132
  IGC: 7
  ND: 4

familles orthologues presentes a la fois chez mais ET riz : 7593 / 12764


In [ ]:
import csv
from collections import defaultdict

OUT_DIR = f"{DATA_DIR}/normalized"

# 1. familles orthologues partagees maïs <-> riz (meme calcul que la validation precedente)
fam_species = defaultdict(set)
orthofam_by_protein = {}
with open(f"{OUT_DIR}/fact_orthology.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        sp = "zea_mays" if row["assemblage"] == "zea_mays" else "oryza_sativa"
        fam_species[row["orthofam_id"]].add(sp)
        orthofam_by_protein[(row["assemblage"], row["protein_accession"])] = row["orthofam_id"]

shared_orthofam = {fam for fam, sps in fam_species.items() if len(sps) == 2}
print(f"familles orthologues partagees maïs/riz : {len(shared_orthofam)}")

# 2. regrouper les annotations fonctionnelles par proteine
pfam_by_protein = defaultdict(list)      # (assemblage, protein) -> [(term_id, term_desc)]
go_quality_by_protein = defaultdict(set)  # (assemblage, protein) -> {evidences non-IEA}

with open(f"{OUT_DIR}/fact_functional_annotation.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        key = (row["assemblage"], row["protein_accession"])
        if row["source"] == "pfam":
            pfam_by_protein[key].append((row["term_id"], row["term_desc"]))
        elif row["source"] == "go_plaza" and row["evidence"] and row["evidence"] != "IEA":
            go_quality_by_protein[key].add(row["evidence"])

# 3. candidats : proteines avec le domaine NB-ARC (PF00931), signature des genes R
candidates = []
for key, domains in pfam_by_protein.items():
    has_nbarc = any(term_id.startswith("PF00931") for term_id, _ in domains)
    if not has_nbarc:
        contiinue
    has_lrr = any("LRR" in (desc or "").upper() for _, desc in domains)
    has_good_go = len(go_quality_by_protein.get(key, set())) > 0
    orthofam = orthofam_by_protein.get(key, "")
    is_conserved = orthofam in shared_orthofam

    score = 2  # NB-ARC present (critere de base pour etre candidat)
    score += 1 if has_lrr else 0
    score += 2 if has_good_go else 0
    score += 1 if is_conserved else 0

    candidates.append({
        "assemblage": key[0],
        "protein_accession": key[1],
        "has_lrr": has_lrr,
        "go_evidences": ",".join(sorted(go_quality_by_protein.get(key, set()))),
        "orthofam_id": orthofam,
        "conserved_maize_rice": is_conserved,
        "score": score,
    })

print(f"\ncandidats avec domaine NB-ARC : {len(candidates)}")

candidates.sort(key=lambda c: c["score"], reverse=True)
print("\ntop 20 candidats :")
for c in candidates[:20]:
    print(c)

# repartition des scores
from collections import Counter
score_dist = Counter(c["score"] for c in candidates)
print("\nrepartition des scores :")
for s, n in sorted(score_dist.items(), reverse=True):
    print(f"  score {s}: {n} candidats")

familles orthologues partagees maïs/riz : 7593

candidats avec domaine NB-ARC : 2477

top 20 candidats :
{'assemblage': 'zea_mays', 'protein_accession': 'XP_008661497.1', 'has_lrr': True, 'go_evidences': 'ISO', 'orthofam_id': 'ORTHO05M001605', 'conserved_maize_rice': True, 'score': 6}
{'assemblage': 'zea_mays', 'protein_accession': 'XP_008674947.1', 'has_lrr': True, 'go_evidences': 'ISO', 'orthofam_id': 'ORTHO05M000177', 'conserved_maize_rice': True, 'score': 6}
{'assemblage': 'oryza_sativa', 'protein_accession': 'XP_015621141.1', 'has_lrr': True, 'go_evidences': 'ISO', 'orthofam_id': 'ORTHO05M000177', 'conserved_maize_rice': True, 'score': 6}
{'assemblage': 'oryza_sativa', 'protein_accession': 'XP_025878943.1', 'has_lrr': True, 'go_evidences': 'IDA,IPI', 'orthofam_id': 'ORTHO05M004057', 'conserved_maize_rice': True, 'score': 6}
{'assemblage': 'oryza_sativa', 'protein_accession': 'XP_025878942.1', 'has_lrr': True, 'go_evidences': 'IDA,IPI', 'orthofam_id': 'ORTHO05M004057', 'conserved_m

In [9]:
import csv
from collections import defaultdict

TOP_CANDIDATES = [
    "XP_008661497.1", "XP_008674947.1",  # zea_mays
    "XP_015621141.1", "XP_025878943.1", "XP_025878942.1", "XP_025878940.1",
    "XP_025878945.1", "XP_025878941.1", "XP_025878944.1", "XP_015629524.2",
    "XP_015645210.1", "XP_015651198.1", "XP_015651196.1", "XP_015651201.1",
    "XP_015651200.1", "XP_015651197.1", "XP_025875888.1", "XP_015651199.1",
    "XP_015612087.1", "XP_015612085.1",  # oryza_sativa
]

profiles = defaultdict(dict)
with open(f"{DATA_DIR}/normalized/fact_expression.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["protein_accession"] in TOP_CANDIDATES:
            profiles[row["protein_accession"]][row["tissue"]] = float(row["tpm_median"])

for acc in TOP_CANDIDATES:
    prof = profiles.get(acc)
    if not prof:
        print(f"{acc}: AUCUNE donnee d'expression trouvee")
        continue
    values = list(prof.values())

    max_tissue = max(prof, key=prof.get)
    print(f"\n{acc} — {len(prof)} tissus")
    print(f"  TPM median: {sum(values)/len(values):.2f} | max: {prof[max_tissue]:.2f} ({max_tissue}) | min: {min(values):.2f}")
    top3 = sorted(prof.items(), key=lambda x: -x[1])[:3]
    print(f"  top 3 tissus : {top3}")


XP_008661497.1 — 23 tissus
  TPM median: 23.35 | max: 52.00 (germinated embryo 2 DAI) | min: 0.40
  top 3 tissus : [('germinated embryo 2 DAI', 52.0), ('root cortex', 50.0), ('mature leaf 8', 45.0)]

XP_008674947.1 — 23 tissus
  TPM median: 8.17 | max: 16.00 (internode 7-8) | min: 0.00
  top 3 tissus : [('internode 7-8', 16.0), ('growth zone of leaf 8', 13.0), ('internode 6-7', 12.0)]

XP_015621141.1 — 7 tissus
  TPM median: 6.57 | max: 11.00 (leaf) | min: 3.00
  top 3 tissus : [('leaf', 11.0), ('root', 10.0), ('post-flowering panicle', 9.0)]

XP_025878943.1 — 7 tissus
  TPM median: 15.14 | max: 28.00 (root) | min: 3.00
  top 3 tissus : [('root', 28.0), ('leaf', 27.0), ('shoot', 17.0)]

XP_025878942.1 — 7 tissus
  TPM median: 15.14 | max: 28.00 (root) | min: 3.00
  top 3 tissus : [('root', 28.0), ('leaf', 27.0), ('shoot', 17.0)]

XP_025878940.1 — 7 tissus
  TPM median: 15.14 | max: 28.00 (root) | min: 3.00
  top 3 tissus : [('root', 28.0), ('leaf', 27.0), ('shoot', 17.0)]

XP_02587894

In [11]:
import csv

CHECK_ACCESSIONS = [
    "XP_025878943.1", "XP_025878942.1", "XP_025878940.1",
    "XP_025878945.1", "XP_025878941.1", "XP_025878944.1",
    "XP_015651198.1", "XP_015651196.1", "XP_015651201.1",
    "XP_015651200.1", "XP_015651197.1", "XP_025875888.1", "XP_015651199.1",
    "XP_015612087.1", "XP_015612085.1",
]

with open(f"{DATA_DIR}/normalized/dim_protein.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["protein_accession"] in CHECK_ACCESSIONS:
            print(row["protein_accession"], "->", row["plaza_gene_id"])

XP_025878943.1 -> Os02g0137800
XP_025878942.1 -> Os02g0137800
XP_025878940.1 -> Os02g0137800
XP_025878945.1 -> Os02g0137800
XP_025878941.1 -> Os02g0137800
XP_025878944.1 -> Os02g0137800
XP_015651198.1 -> Os09g0267800
XP_015651196.1 -> Os09g0267800
XP_015651201.1 -> Os09g0267800
XP_015651200.1 -> Os09g0267800
XP_015651197.1 -> Os09g0267800
XP_025875888.1 -> Os09g0267800
XP_015651199.1 -> Os09g0267800
XP_015612087.1 -> Os09g0272900
XP_015612085.1 -> Os09g0272900


In [13]:
import csv
from collections import defaultdict

# recharge le pont protein_accession -> plaza_gene_id (cle de dedup au niveau gene)
protein_to_gene = {}
with open(f"{DATA_DIR}/normalized/dim_protein.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        key = (row["assemblage"], row["protein_accession"])
        # pour le tabac (pas de plaza_gene_id), le protein_accession fait office de cle gene
        protein_to_gene[key] = row.get("plaza_gene_id") or row["protein_accession"]

annotations = defaultdict(list)
with open(f"{DATA_DIR}/normalized/fact_functional_annotation.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        annotations[(row["assemblage"], row["protein_accession"])].append(row)

orthofam_by_protein = {}
fam_species = defaultdict(set)
with open(f"{DATA_DIR}/normalized/fact_orthology.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        key = (row["assemblage"], row["protein_accession"])
        orthofam_by_protein[key] = row["orthofam_id"]
        sp = "zea_mays" if row["assemblage"] == "zea_mays" else "oryza_sativa"
        fam_species[row["orthofam_id"]].add(sp)

shared_orthofam = {fam for fam, sps in fam_species.items() if len(sps) == 2}

# meme scoring qu'avant, mais on garde seulement le meilleur score par gene (plaza_gene_id)
best_per_gene = {}
for key, rows in annotations.items():
    has_nbarc = any(r["source"] == "pfam" and r["term_id"].startswith("PF00931") for r in rows)
    if not has_nbarc:
        continue
    has_lrr = any("LRR" in (r["term_desc"] or "").upper() for r in rows if r["source"] == "pfam")
    has_good_go = any(r["source"] == "go_plaza" and r["evidence"] and r["evidence"] != "IEA" for r in rows)
    is_conserved = orthofam_by_protein.get(key, "") in shared_orthofam

    score = 2 + (1 if has_lrr else 0) + (2 if has_good_go else 0) + (1 if is_conserved else 0)
    gene_key = (key[0], protein_to_gene.get(key, key[1]))
    if gene_key not in best_per_gene or score > best_per_gene[gene_key]["score"]:
        best_per_gene[gene_key] = {"score": score, "example_protein": key[1]}

print(f"genes candidats distincts (dedup par plaza_gene_id) : {len(best_per_gene)}")
from collections import Counter
dist = Counter(v["score"] for v in best_per_gene.values())
for s, n in sorted(dist.items(), reverse=True):
    print(f"  score {s}: {n} genes")

genes candidats distincts (dedup par plaza_gene_id) : 2104
  score 6: 15 genes
  score 5: 33 genes
  score 4: 131 genes
  score 3: 1356 genes
  score 2: 569 genes
